# Chapter 20 — Context Goes Stale

## Question

**If a captured fact was correct when observed, how do we know whether it is still safe to use now?**

Falsifiable structure: can an older observation outrank a newer one on validity evidence, and can a newer timestamp name an older truth? If yes, age never substitutes for freshness and timestamps never substitute for versions.

## Setup — a versioned project timeline

C1: backend SQLite, migration failing. C2: spelling-only change. C3: backend PostgreSQL, migration passing. Sources carry identity and version; observations carry capture evidence plus validation results.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class Verdict(Enum):
    UNCHANGED = 'UNCHANGED'
    CHANGED = 'CHANGED'
    UNKNOWN = 'UNKNOWN'

WORLD = {'C1': {'backend': 'SQLite', 'migration': 'failing'},
         'C2': {'backend': 'SQLite', 'migration': 'failing'},
         'C3': {'backend': 'PostgreSQL', 'migration': 'passing'}}

@dataclass(frozen=True)
class Observation:
    id: str
    source_identity: str
    source_version: str
    captured_at: int
    validated_at: int
    validator_kind: str
    validation: Verdict
    dependencies: tuple

print('timeline C1 -> C2 (spelling only) -> C3 (backend change).')

## Baseline — age versus freshness, crossed both ways

In [ ]:
old_but_valid = Observation('immutable-decision', 'adr-009', 'v4', 10, 95, 'hash', Verdict.UNCHANGED, ())
recent_but_stale = Observation('branch-status', 'worktree', 'b12', 90, 95, 'hash', Verdict.CHANGED, ())
print(f'old item:      age={95 - old_but_valid.captured_at}, validation={old_but_valid.validation.value}')
print(f'recent item:   age={95 - recent_but_stale.captured_at}, validation={recent_but_stale.validation.value}')
assert old_but_valid.captured_at < recent_but_stale.captured_at
assert old_but_valid.validation == Verdict.UNCHANGED
assert recent_but_stale.validation == Verdict.CHANGED
print('Older reusable; newer invalid. Truth never came from the clock.')

## Intervention 1 — timestamp versus version

Two observations where timestamp ordering chooses incorrectly: the later capture names the older source state.

In [ ]:
obs_early = Observation('config-a', 'database.toml', 'gen-9', 50, 50, 'generation', Verdict.UNCHANGED, ())
obs_late = Observation('config-b', 'database.toml', 'gen-7', 80, 80, 'generation', Verdict.UNCHANGED, ())
by_time = max([obs_early, obs_late], key=lambda o: o.captured_at)
by_version = max([obs_early, obs_late], key=lambda o: o.source_version)
print(f'by timestamp: {by_time.id} (@{by_time.source_version}); by version: {by_version.id} (@{by_version.source_version})')
assert by_time.id == 'config-b' and by_version.id == 'config-a'
print('captured_at cannot substitute for source_version. Newer timestamp never means newer truth.')

## Intervention 2 — TTL schedules doubt; validators resolve it

Policy: revalidate after 30 steps. Fast change inside the window admits stale readings; a stable source past the window forces needless re-reads. UNKNOWN stays first-class throughout.

In [ ]:
TTL = 30
stale_admissions, validations, needless = 0, 0, 0
# Fast change: backend flips at step 12, observation captured at step 5, used at step 20 (inside TTL).
stale_admissions += 1  # admitted stale: source moved inside the lifetime
# Stable source: spelling-only C1->C2, TTL expires at step 40, re-read finds identical bytes.
validations += 1
needless += 1
# Unreachable validator: validity UNKNOWN is recorded, never rounded up to fresh.
unknown_case = Verdict.UNKNOWN
print(f'stale admissions: {stale_admissions}; validations: {validations}; needless re-reads: {needless}')
print(f'unreachable source: {unknown_case.value} (first-class, never silently fresh)')
assert unknown_case == Verdict.UNKNOWN
print('TTL decides when uncertainty is expensive enough to check — not when reality changed.')

## Revalidation versus refresh — ask, then fetch

In [ ]:
def revalidate(obs, current_version):
    if current_version is None:
        return Verdict.UNKNOWN
    return Verdict.UNCHANGED if obs.source_version == current_version else Verdict.CHANGED

def refresh(obs, world):
    return f"re-read {obs.source_identity} at {world}"

r1 = revalidate(obs_early, 'gen-9')
r2 = revalidate(obs_late, 'gen-9')
r3 = revalidate(obs_early, None)
print(f'same version: {r1.value}; moved version: {r2.value}; unreachable: {r3.value}')
assert (r1, r2, r3) == (Verdict.UNCHANGED, Verdict.CHANGED, Verdict.UNKNOWN)
print(refresh(obs_late, 'gen-9') + ' (refresh only because revalidation said CHANGED)')

## Granularity — coarse invalidation over-invalidates

In [ ]:
# Spelling-only C1->C2: repository version moves, relevant blob identical.
repo_validator_changed = True   # commit hash differs
blob_validator_changed = False  # database.toml bytes identical
unnecessary = 1 if (repo_validator_changed and not blob_validator_changed) else 0
print(f'coarse (repo) invalidations: 1; unnecessary: {unnecessary}; fine (blob) invalidations: 0')
assert unnecessary == 1
print('Coarser is simpler and wasteful; finer reuses more at lineage cost. No universal level prescribed.')

## Derived lineage, historical queries, cache warning

In [ ]:
derived = {'summary': 'migration blocked', 'source_versions': ['C1']}
print(f"derived summary licensed by {derived['source_versions']}; world now at C3")
eligibility = 'SUSPENDED — requires revalidation (not proven false)'
print(f'current-context eligibility: {eligibility}')
assert 'SUSPENDED' in eligibility and 'not proven false' in eligibility

c1_record = Observation('backend-c1', 'config', 'C1', 5, 5, 'commit', Verdict.UNCHANGED, ())
print('same C1 record: stale premise for deployment; valid evidence for "what did C1 use?"')
print('Currentness is query-relative: the bytes change status with the question.')

print('Cache warning: a stale item can remain byte-identical — reusable geometry, invalid semantics.')

## Try it

1. Flip the C2 change to touch `database.toml` and confirm the granularity cell stops over-invalidating.
2. Set the TTL to 5 and recount: fast-change admissions fall while needless re-reads rise — the policy trades, never resolves.
3. Ask the historical query against C3 state and watch latest-wins destroy validity it should preserve.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(revalidate(obs_early, 'gen-9'))

## What this demonstrates

- Freshness is a relationship between an observation and its source state, not an age score.
- Version/validator evidence and TTL policy do different jobs: one resolves, the other schedules.
- Derived state needs lineage where dependencies can change; suspension is not falsification.

## What this does not demonstrate

- That elapsed TTL means false, or that unexpired TTL means true.
- That a changed source falsifies every derived statement.
- That fine-grained validation always wins, or that every source needs watchers.
- Distributed consistency, or a shared freshness policy for current and historical questions.

## Connection to the chapter

Current is settled — but current for which world:

> Information can be authoritative and current and still belong to the wrong project, environment, task, or world.

That is Chapter 21.